<a href="https://colab.research.google.com/github/charlottesscott/Breast_Screening_Uptake_Prediction/blob/main/Population_age.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Libaries**

In [2]:
# Install xlrd so Colab can read the older .xls file format
!pip install xlrd -q
import pandas as pd

**Upload CSV**

In [3]:
from google.colab import files

uploaded_files = files.upload()

Saving popprojla5yr5year22based.xls to popprojla5yr5year22based.xls


**Read CSV**

In [5]:
df = pd.read_excel("popprojla5yr5year22based.xls")

**Read Females and Persons Sheet**

In [7]:
# ---------------------------------------------------------
# 2. Read the Females and Persons sheets
# ---------------------------------------------------------
# The workbook has three title rows, so header=3 uses row 4
# as the column headings.
females = pd.read_excel(
"popprojla5yr5year22based.xls",
sheet_name="Females",
header=3,
engine="xlrd"
)
persons = pd.read_excel(
"popprojla5yr5year22based.xls",
sheet_name="Persons",
header=3,
engine="xlrd"
)

**Standardise Column Names**

In [10]:
def clean_columns(data):
    data = data.copy()
    data.columns = [
        int(column) if isinstance(column, float) and column.is_integer()
        else column
        for column in data.columns
    ]
    return data

    females = clean_columns(females)
persons = clean_columns(persons)

**Check for columns**

In [12]:
# ---------------------------------------------------------
# 4. Check that the required columns are available
# ---------------------------------------------------------
required_columns = [
    "CODE",
    "AREA",
    "AGE GROUP",
    2022
]
for sheet_name, data in {
    "Females": females,
    "Persons": persons
}.items():
    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]
    if missing_columns:
        raise ValueError(
            f"{sheet_name} is missing these columns: "
            f"{missing_columns}"
        )

In [16]:
df.head()

,"Population projections for local authorities by five-year age groups and sex, England 2022-based"
0,Date published: 24 June 2025
1,Coverage: Local authorities and higher adminis...
2,Projection: Five-year migration variant
3,"This spreadsheet contains 6 sheets, 3 of which..."
4,Publication Details


**Calculating population estimates 53 to 71**

In [14]:
# ---------------------------------------------------------
# 5. Calculate the estimated population aged 53 to under 71
# ---------------------------------------------------------
# This follows the requested approach:
#
# 50-54: use 1/2
# 55-59: use all
# 60-64: use all
# 65-69: use all
# 70-74: use 1/4

age_group_weights = {
    "50-54": 0.50,
    "55-59": 1.00,
    "60-64": 1.00,
    "65-69": 1.00,
    "70-74": 0.25
}

def calculate_adjusted_population(data, output_column):
    selected = data[
        data["AGE GROUP"].isin(age_group_weights)
    ].copy()

    selected["Age group weight"] = (
        selected["AGE GROUP"].map(age_group_weights)
    )
    selected["Adjusted population"] = (
        selected[2022] * selected["Age group weight"]
    )
    summary = (
        selected
        .groupby(
            ["CODE", "AREA"],
            as_index=False
        )["Adjusted population"]
        .sum()
        .rename(
            columns={
                "Adjusted population": output_column
            }
        )
    )
    return summary

female_summary = calculate_adjusted_population(
    females,
    "Female population aged 53 to under 71"
)
persons_summary = calculate_adjusted_population(
    persons,
    "Persons population aged 53 to under 71"
)

In [15]:
df.head()

,"Population projections for local authorities by five-year age groups and sex, England 2022-based"
0,Date published: 24 June 2025
1,Coverage: Local authorities and higher adminis...
2,Projection: Five-year migration variant
3,"This spreadsheet contains 6 sheets, 3 of which..."
4,Publication Details


**Join results together**

In [17]:
# ---------------------------------------------------------
# 6. Join the female and persons results using the area code
# ---------------------------------------------------------
final_data = female_summary.merge(
persons_summary[
[
"CODE",
"Persons population aged 53 to under 71"
]
],
on="CODE",
how="inner",
validate="one_to_one"
)

**Calculate Female Percentage**

In [18]:
# ---------------------------------------------------------
# 7. Calculate the female percentage
# ---------------------------------------------------------
# Female percentage =
# female population aged 53 to under 71
# divided by
# persons population aged 53 to under 71
final_data["Female percentage"] = (
final_data["Female population aged 53 to under 71"]
/
final_data["Persons population aged 53 to under 71"]
) * 100

**Round output**

In [19]:
population_columns = [
"Female population aged 53 to under 71",
"Persons population aged 53 to under 71"
]
final_data[population_columns] = (
final_data[population_columns]
.round(0)
.astype("Int64")
)
final_data["Female percentage"] = (
final_data["Female percentage"].round(2)
)

**Sort by area name**

In [20]:
final_data = (
final_data
.sort_values("AREA")
.reset_index(drop=True)
)

**Display**

In [21]:
display(final_data.head(20))
print(f"Number of areas in output: {len(final_data):,}")
print("\nMissing values:")
print(final_data.isna().sum())

,CODE,AREA,Female population aged 53 to under 71,Persons population aged 53 to under 71,Female percentage
0,E07000223,Adur,8023,15542,51.62
1,E07000026,Allerdale,13676,26884,50.87
2,E07000032,Amber Valley,16841,33304,50.57
3,E07000224,Arun,22658,43197,52.45
4,E07000170,Ashfield,15359,30212,50.84
5,E07000105,Ashford,15922,31114,51.17
6,E07000200,Babergh,13064,25182,51.88
7,E09000002,Barking and Dagenham,17794,34899,50.99
8,E09000003,Barnet,39036,75186,51.92
9,E08000016,Barnsley,30647,60390,50.75


Number of areas in output: 349

Missing values:
CODE                                      0
AREA                                      0
Female population aged 53 to under 71     0
Persons population aged 53 to under 71    0
Female percentage                         0
dtype: int64
